In [9]:
%load_ext autoreload 
%autoreload 2

In [ ]:
from typing import Tuple, List, Any

In [10]:
import random
from xgboost import XGBClassifier, XGBRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
import sklearn
import numpy as np 

random.seed(42)
np.random.seed(42)

Step 1. Loading Data

In [ ]:
import sys
import os

# Add project root (parent of "src") to Python path
sys.path.append(os.path.abspath(".."))

from src.data_loader import load_OhioT1DM

Ohio_PATH = "..\data\\raw\\"
X_train, X_val, X_test, y_reg_train, y_reg_val, y_reg_test, y_clf_train, y_clf_val, y_clf_test = load_OhioT1DM(path=Ohio_PATH, look_back=64)

In [33]:
LBW = X_train.shape[1]
PH = y_reg_train.shape[1]

Step 2. Define Model

In [37]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 

In [ ]:
import math

def conv1d_output_length(L_in, kernel_size, stride=2, padding=1, dilation=1):
    """
    Compute the output length of a 1D convolution layer.

    Parameters
    ----------
    L_in : int
        Input length (sequence length).
    kernel_size : int
        Size of the convolution kernel.
    stride : int, default=1
        Convolution stride.
    padding : int, default=0
        Padding on both sides.
    dilation : int, default=1
        Dilation factor.

    Returns
    -------
    L_out : int
        Output length after the convolution.
    """
    return math.floor((L_in + 2*padding - dilation*(kernel_size - 1) - 1) / stride + 1)

In [ ]:
class ConvBlock(nn.Module): # DCGAN inspired
    def __init__(self, in_channels, out_channels, kernel_size, down=True, use_bn=True):
        super().__init__()
        if down:
            self.block = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size, 2, 1, bias=False),
                nn.BatchNorm1d(out_channels) if use_bn else nn.Identity(),
                nn.LeakyReLU(0.2, inplace=True)
            )
        else:
            self.block = nn.Sequential(
                nn.ConvTranspose1d(in_channels, out_channels, kernel_size, 2, 1, 1, bias=False),
                nn.BatchNorm1d(out_channels) if use_bn else nn.Identity(),
                nn.ReLU(inplace=True)
            )
    def forward(self, x):
        return self.block(x)



class VAE(nn.Module): 
    
    def __init__(self, input_shape : Tuple[int, int], latent_dim: int , hidden_dims:List[int] = None, kernel_sizes:List[int] = None):
        super(VAE, self).__init__( )
        
        if hidden_dims is None : 
            hidden_dims = [16, 32, 64]
        
        if kernel_sizes is None : 
            kernel_sizes = [5, 4, 3]
        
        # build encoder    
        blocks = []
        in_channels = input_shape[1]
        L_in = input_shape[0]
        for hidden_dim, kernel_size in zip(hidden_dims, kernel_sizes): 
            
            blocks.append(ConvBlock(in_channels, hidden_dim, kernel_size))
            in_channels = hidden_dim
            
            L_out = conv1d_output_length(L_in, kernel_size)
            L_in = L_out
            
        self.encoder = nn.Sequential(*blocks)
            
        self.mu = nn.Linear(L_in * hidden_dims[-1], latent_dim) 
        self.log_var = nn.Linear(L_in * hidden_dims[-1], latent_dim) 
        
        kernel_sizes.reverse()
        hidden_dims.reverse() 
        
        self.decoder_input = nn.Linear(latent_dim, L_in * hidden_dims[0])
        
        # build decoder    
        blocks = []
        in_channels = input_shape[1]
        L_out = input_shape[0]
        for hidden_dim, kernel_size in zip(hidden_dims, kernel_sizes): 
            
            blocks.append(ConvBlock(in_channels, hidden_dim, kernel_size, down=False))
            in_channels = hidden_dim
            
        self.decoder = nn.Sequential(*blocks) 
        
        self.final_layer = nn.Sequential(
            ConvBlock(hidden_dims[-1], hidden_dims[-1], kernel_size=kernel_sizes[-1], down=False), 
            nn.Conv1d(hidden_dims[-1], input_shape[1], 3, padding=1), 
        )
            
        
        
        